# Bankroll Simulation: Optimal min_edge for Compounding

Loads trades from the KDE backtest and simulates bankroll trajectories for different min_edge thresholds and risk fractions.
Treats all positions within a movie as perfectly correlated (worst case).

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent
sys.path.insert(0, str(ROOT))

## Re-run backtest to get trades DataFrame

This re-uses the exact same code from `kde_backtest.ipynb`. Takes ~4 min.

In [ ]:
import io, contextlib, time, warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)

from edge import compute_edge
from critic_model import (
    build_critic_profiles, build_kde_lambda_model,
    default_training_slugs, estimate_lambda, estimate_p_fresh,
)

PRICE_DIR = ROOT / "rt-price-histories"

# --- Load data ---
reviews_df = pd.read_csv(ROOT / "reviews.csv")
reviews_df["estimated_timestamp"] = pd.to_datetime(
    reviews_df["estimated_timestamp"], format="ISO8601", utc=True
)
movies_df = pd.read_csv(ROOT / "movies_index.csv")
movies_df["Bet Close Date"] = pd.to_datetime(movies_df["Bet Close Date"], utc=True)

slugs_with_prices = sorted([
    d.name for d in PRICE_DIR.iterdir()
    if d.is_dir() and list(d.glob("*hour*"))
])
movies_bt = movies_df[movies_df["Slug"].isin(slugs_with_prices)].copy()
movies_bt = movies_bt.dropna(subset=["Bet Close Date"]).sort_values("Bet Close Date")

# --- Paste helpers from kde_backtest (kept minimal) ---
def load_hourly_prices(slug):
    csv_files = list((PRICE_DIR / slug).glob("*hour*"))
    if not csv_files: return None
    df = pd.read_csv(csv_files[0])
    df["timestamp"] = pd.to_datetime(df["timestamp"], utc=True)
    df = df.sort_values("timestamp").reset_index(drop=True)
    thresh_cols = [c for c in df.columns if c.startswith("Above ")]
    df[thresh_cols] = df[thresh_cols].ffill()
    return df

def get_resolution(price_df):
    resolution = {}
    for col in [c for c in price_df.columns if c.startswith("Above ")]:
        thresh = int(col.split()[-1])
        last_valid = price_df[col].dropna()
        if last_valid.empty: resolution[thresh] = None; continue
        terminal = last_valid.iloc[-1]
        if terminal >= 90: resolution[thresh] = True
        elif terminal <= 10: resolution[thresh] = False
        else: resolution[thresh] = None
    return resolution

def precompute_review_states(slug, reviews_df, bet_close):
    movie_reviews = reviews_df[reviews_df["movie_slug"] == slug].copy()
    movie_reviews = movie_reviews[movie_reviews["estimated_timestamp"] <= bet_close]
    movie_reviews = movie_reviews.sort_values("estimated_timestamp").reset_index(drop=True)
    if movie_reviews.empty: return [], None
    states, critics, fresh, total = [], set(), 0, 0
    for _, row in movie_reviews.iterrows():
        critics = critics | {row["reviewer_name"]}
        total += 1
        if row["tomatometer_sentiment"] == "positive": fresh += 1
        states.append({"timestamp": row["estimated_timestamp"],
                       "observed_critics": frozenset(critics), "fresh_count": fresh, "total_count": total})
    return states, movie_reviews["estimated_timestamp"].iloc[0]

def get_review_state_at(states, snapshot_time):
    if not states or snapshot_time < states[0]["timestamp"]: return set(), 0, 0
    lo, hi = 0, len(states) - 1
    while lo < hi:
        mid = (lo + hi + 1) // 2
        if states[mid]["timestamp"] <= snapshot_time: lo = mid
        else: hi = mid - 1
    s = states[lo]
    return set(s["observed_critics"]), s["fresh_count"], s["total_count"]

def backtest_movie(slug, reviews_df, movies_df, every_n_hours=24):
    row = movies_df[movies_df["Slug"] == slug].iloc[0]
    bet_close_date = row["Bet Close Date"]
    price_df = load_hourly_prices(slug)
    if price_df is None or price_df.empty: return []
    market_close_time = price_df["timestamp"].iloc[-1]
    resolution = get_resolution(price_df)
    training_slugs = default_training_slugs(movies_df, exclude_slug=slug, before_date=bet_close_date)
    if len(training_slugs) < 5: return []
    buf = io.StringIO()
    with contextlib.redirect_stdout(buf):
        profiles = build_critic_profiles(reviews_df, movies_df, training_slugs)
        model = build_kde_lambda_model(profiles)
    review_states, first_review_ts = precompute_review_states(slug, reviews_df, market_close_time)
    thresh_cols = [c for c in price_df.columns if c.startswith("Above ")]
    records = []
    cached_critics, cached_fresh, cached_total = set(), 0, 0
    cached_lambda, cached_p_fresh = None, None
    last_kept_ts = None
    for i in range(len(price_df)):
        snap_row = price_df.iloc[i]
        snapshot_time = snap_row["timestamp"]
        if every_n_hours > 1 and last_kept_ts is not None:
            if (snapshot_time - last_kept_ts).total_seconds() / 3600 < every_n_hours: continue
        last_kept_ts = snapshot_time
        hours_to_close = (market_close_time - snapshot_time).total_seconds() / 3600
        if hours_to_close <= 0: continue
        days_before_close = hours_to_close / 24
        observed_critics, fresh_count, total_count = get_review_state_at(review_states, snapshot_time)
        state_changed = (total_count != cached_total)
        first_review_dbc = None
        if first_review_ts is not None and (total_count if state_changed else cached_total) > 0:
            first_review_dbc = (market_close_time - first_review_ts).total_seconds() / 86400
        if state_changed or cached_lambda is None:
            cached_critics, cached_fresh, cached_total = observed_critics, fresh_count, total_count
            cached_lambda = estimate_lambda(model, days_before_close, hours_to_close,
                observed_critics, observed_count=total_count, first_review_dbc=first_review_dbc)
            cached_p_fresh = estimate_p_fresh(profiles, observed_critics, fresh_count, total_count)
        else:
            cached_lambda = estimate_lambda(model, days_before_close, hours_to_close,
                cached_critics, observed_count=cached_total, first_review_dbc=first_review_dbc)
        for col in thresh_cols:
            thresh = int(col.split()[-1])
            market_price = snap_row[col]
            if pd.isna(market_price): continue
            resolved = resolution.get(thresh)
            if resolved is None: continue
            try:
                result = compute_edge(threshold=thresh, market_price=market_price,
                    fresh_count=cached_fresh, total_count=cached_total,
                    hours_to_close=hours_to_close, lambda_rate=cached_lambda, p_fresh=cached_p_fresh)
            except: continue
            records.append({"slug": slug, "snapshot_time": snapshot_time, "hours_to_close": hours_to_close,
                "threshold": thresh, "market_price": market_price, "model_p_yes": result["p_yes"],
                "edge_cents": result["edge_cents"], "resolved_yes": resolved,
                "lambda_rate": cached_lambda, "p_fresh": cached_p_fresh,
                "fresh_count": cached_fresh, "total_count": cached_total})
    return records

# --- Run backtest ---
all_records, skipped = [], []
slugs = movies_bt["Slug"].tolist()
t0 = time.time()
for idx, slug in enumerate(slugs):
    elapsed = time.time() - t0
    rate = (idx / elapsed) if elapsed > 0 and idx > 0 else 0
    eta = (len(slugs) - idx) / rate if rate > 0 else 0
    print(f"\r[{idx+1}/{len(slugs)}] {slug:<40s} ({elapsed:.0f}s, ~{eta:.0f}s left)", end="", flush=True)
    try: all_records.extend(backtest_movie(slug, reviews_df, movies_df))
    except Exception as e: skipped.append((slug, str(e)))
print(f"\nDone in {time.time()-t0:.0f}s. {len(all_records):,} evaluations, {len(slugs)-len(skipped)} movies.")

trades = pd.DataFrame(all_records)
trades["direction"] = np.where(trades["edge_cents"] >= 0, "Yes", "No")
trades["abs_edge"] = trades["edge_cents"].abs()
print(f"Shape: {trades.shape}, Movies: {trades['slug'].nunique()}")

## Bankroll simulation

In [ ]:
def simulate_bankroll(trades_df, min_edge, bankroll_frac, start_bankroll=100000.0):
    """Simulate bankroll growth for a No-only strategy.
    
    Treats all positions within a movie as perfectly correlated (worst case).
    Sizes each movie's total risk as bankroll_frac of current bankroll.
    
    Args:
        start_bankroll: in cents (100000 = $1,000)
        bankroll_frac: fraction of bankroll to risk per movie
    """
    ACTION_WINDOW = (24, 120)
    mask = (
        (trades_df["direction"] == "No") &
        (trades_df["abs_edge"] >= min_edge) &
        (trades_df["hours_to_close"] >= ACTION_WINDOW[0]) &
        (trades_df["hours_to_close"] <= ACTION_WINDOW[1])
    )
    no_trades = trades_df[mask].sort_values("snapshot_time")
    positions = no_trades.groupby(["slug", "threshold"]).first().reset_index()
    positions["entry_cost"] = 100 - positions["market_price"]
    positions["pos_pnl"] = np.where(
        ~positions["resolved_yes"],
        positions["market_price"],
        -positions["entry_cost"],
    )
    
    # Aggregate to movie level, ordered by first entry time
    movie_results = positions.groupby("slug").agg(
        first_entry=("snapshot_time", "min"),
        total_pnl=("pos_pnl", "sum"),
        total_cost=("entry_cost", "sum"),
        n_positions=("pos_pnl", "count"),
    ).sort_values("first_entry").reset_index()
    
    # Simulate bankroll trajectory
    bankroll = start_bankroll
    trajectory = [{"movie": "START", "bankroll": bankroll, "time": movie_results["first_entry"].min()}]
    
    for _, movie in movie_results.iterrows():
        risk_budget = bankroll * bankroll_frac
        # Scale: how many contracts fit the risk budget across all positions
        contracts_scale = risk_budget / movie["total_cost"] if movie["total_cost"] > 0 else 0
        movie_pnl = movie["total_pnl"] * contracts_scale
        bankroll += movie_pnl
        if bankroll <= 0:
            bankroll = 0
            trajectory.append({"movie": movie["slug"], "bankroll": 0, "time": movie["first_entry"]})
            break
        trajectory.append({"movie": movie["slug"], "bankroll": bankroll, "time": movie["first_entry"]})
    
    return pd.DataFrame(trajectory), movie_results

In [ ]:
START_BANKROLL = 100000  # $1,000 in cents
BANKROLL_FRAC = 0.10     # 10% of bankroll per movie

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Left: vary min_edge at fixed risk fraction
ax = axes[0]
for me in [5, 10, 15, 20]:
    traj, _ = simulate_bankroll(trades, me, BANKROLL_FRAC, START_BANKROLL)
    ax.plot(traj["time"], traj["bankroll"] / 100, label=f"min_edge={me}c", linewidth=1.2)
ax.axhline(START_BANKROLL / 100, color="k", linewidth=0.3, linestyle="--")
ax.set_xlabel("Time")
ax.set_ylabel("Bankroll ($)")
ax.set_title(f"Bankroll growth by min_edge (risk {BANKROLL_FRAC:.0%}/movie)")
ax.legend()
ax.tick_params(axis="x", rotation=30)

# Right: vary risk fraction at fixed min_edge
ax = axes[1]
FIXED_EDGE = 10
for bf in [0.05, 0.10, 0.15, 0.20]:
    traj, _ = simulate_bankroll(trades, FIXED_EDGE, bf, START_BANKROLL)
    ax.plot(traj["time"], traj["bankroll"] / 100, label=f"risk={bf:.0%}/movie", linewidth=1.2)
ax.axhline(START_BANKROLL / 100, color="k", linewidth=0.3, linestyle="--")
ax.set_xlabel("Time")
ax.set_ylabel("Bankroll ($)")
ax.set_title(f"Bankroll growth by risk fraction (min_edge={FIXED_EDGE}c)")
ax.legend()
ax.tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.show()

In [ ]:
# Summary table: all combinations
print(f"Starting bankroll: ${START_BANKROLL/100:,.0f}")
print(f"\n{'min_edge':>10s} {'risk%':>6s} {'final $':>10s} {'multiplier':>12s} {'movies':>7s}")
print("-" * 50)
for me in [5, 10, 15, 20]:
    for bf in [0.05, 0.10, 0.15]:
        traj, mr = simulate_bankroll(trades, me, bf, START_BANKROLL)
        final = traj["bankroll"].iloc[-1]
        mult = final / START_BANKROLL
        print(f"{me:>8d}c {bf:>5.0%} ${final/100:>10,.0f} {mult:>11.1f}x {len(mr):>7d}")

# Best compounding configuration
print("\n=== Optimal for compounding ===")
best_mult = 0
best_config = None
for me in [5, 10, 15, 20]:
    for bf in [0.05, 0.10, 0.15, 0.20, 0.25]:
        traj, mr = simulate_bankroll(trades, me, bf, START_BANKROLL)
        final = traj["bankroll"].iloc[-1]
        mult = final / START_BANKROLL
        # Check if bankroll ever went below 50% of start (risk of ruin proxy)
        min_bankroll = traj["bankroll"].min()
        drawdown = 1 - min_bankroll / START_BANKROLL
        if mult > best_mult:
            best_mult = mult
            best_config = (me, bf, mult, len(mr), drawdown)
        if me == 10:  # Print the full grid for min_edge=10
            print(f"  min_edge=10c, risk={bf:.0%}: {mult:.1f}x, max drawdown={drawdown:.0%}")

print(f"\nBest overall: min_edge={best_config[0]}c, risk={best_config[1]:.0%} "
      f"-> {best_config[2]:.1f}x over {best_config[3]} movies (max drawdown={best_config[4]:.0%})")

## Temporal distribution of bets

Are the movies that trigger bets spread evenly across time, or clustered?

In [ ]:
ACTION_WINDOW = (24, 120)

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle("Temporal distribution of movies triggering bets (No-only, T-1d to T-5d)", fontsize=14)

for ax, me in zip(axes.flat, [5, 10, 15, 20]):
    mask = (
        (trades["direction"] == "No") &
        (trades["abs_edge"] >= me) &
        (trades["hours_to_close"] >= ACTION_WINDOW[0]) &
        (trades["hours_to_close"] <= ACTION_WINDOW[1])
    )
    no_trades = trades[mask].sort_values("snapshot_time")
    movie_entries = no_trades.groupby("slug")["snapshot_time"].min().sort_values()

    ax.scatter(movie_entries.values, range(len(movie_entries)), s=12, alpha=0.7)
    ax.set_title(f"min_edge={me}c ({len(movie_entries)} movies)")
    ax.set_xlabel("First bet entry time")
    ax.set_ylabel("Cumulative movie count")
    ax.tick_params(axis="x", rotation=30)

    if len(movie_entries) > 1:
        diffs = pd.Series(movie_entries.values).diff().dropna()
        max_gap = diffs.max()
        median_gap = diffs.median()
        max_gap_idx = diffs.idxmax()
        before_slug = movie_entries.index[max_gap_idx - 1]
        after_slug = movie_entries.index[max_gap_idx]
        before_date = movie_entries.iloc[max_gap_idx - 1]
        after_date = movie_entries.iloc[max_gap_idx]
        ax.text(0.02, 0.95, f"median gap: {median_gap.days}d\nmax gap: {max_gap.days}d",
                transform=ax.transAxes, va="top", fontsize=9,
                bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.5))
        print(f"min_edge={me}c: max gap = {max_gap.days}d between "
              f"{before_slug} ({before_date.strftime('%Y-%m-%d')}) and "
              f"{after_slug} ({after_date.strftime('%Y-%m-%d')})")

plt.tight_layout()
plt.show()